# Kredi Kartı Dolandırıcılığı Tespiti

Önceki egzersizler, bir sinir ağının tüm farklı bileşenlerine daha yakından bakmanızı sağladı:
* sıralı (sequential) yoğun (Dense) bir Sinir Ağı mimarisi,
* derleme (compilation) yöntemi,
* modelin eğitilmesi (fitting).

Şimdi **çok fazla veri içeren** gerçek hayat bir veri seti üzerinde çalışalım!

**Veri seti: `Kredi Kartı İşlemleri (Credit Card Transactions)`**

Bu açık uçlu challenge’da, **kredi kartı işlemlerinden elde edilmiş verilerle** çalışacaksınız.

Bu veriler **hassas** olduğu için, toplam 31 sütundan yalnızca 3’ü bilinmektedir; geri kalanlar verileri **anonimleştirmek** amacıyla dönüştürülmüştür (aslında bunlar, orijinal verilerin **PCA projeksiyonlarıdır**).

Bilinen 3 sütun şunlardır:

* `TIME`: İşlemin, veri setindeki ilk işleme göre geçen süresi  
* `AMOUNT`: İşlem tutarı  
* `CLASS` (hedef değişkenimiz):
    * `0 : geçerli işlem`
    * `1 : sahte (fraud) işlem`

❓ **Soru** ❓ Veri setini indirerek başlayın:
* Kaggle üzerinden: [buradan](https://www.kaggle.com/mlg-ulb/creditcardfraud)
* veya bizim URL’mizden: [buradan](https://d32aokrjazspmn.cloudfront.net/materials/creditcard.csv)

Veriyi yükleyerek `X` ve `y` değişkenlerini oluşturun.

In [6]:
import pandas as pd

# Doğrudan URL'den yükle (Kaggle hesabı gerekmez)
url = "https://d32aokrjazspmn.cloudfront.net/materials/creditcard.csv"
df = pd.read_csv(url)

print("First 5 records:", df.head())
print("Shape:", df.shape)

# X ve y oluştur
X = df.drop(columns=["Class"])
y = df["Class"]


First 5 records:    Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26    

## 1. Sınıfların yeniden dengelenmesi

In [7]:
# Sınıf dengesini kontrol edelim
pd.Series(y).value_counts(normalize=True)

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

☝️ Bu `fraud detection` (sahtekârlık tespiti) challenge’ında **sınıflar aşırı derecede dengesizdir**:
* %99.8 normal işlemler
* %0.2 sahte (fraud) işlemler

**Ciddi yeniden dengeleme (rebalancing) stratejileri uygulamazsak sahtekârlık vakalarını tespit edemeyiz!**

❓ **Soru** ❓
1. **Önce**, veri setinizden üç ayrı bölme oluşturun: `Train / Val / Test`.  
   Doğrulama (validation) ve test setlerinin **dengesiz** kalması son derece önemlidir; böylece modeli değerlendirirken gerçek koşullar korunur ve veri sızıntısı (data leakage) oluşmaz. Test setinizi bu notebook’un **en son hücresine kadar saklayın**!

&nbsp;

2. **İkinci olarak**, yalnızca eğitim setinizi (training set) yeniden dengeleyin. Birçok seçeneğiniz var:

- Azınlık sınıfını rastgele oversample etmek (düz NumPy fonksiyonlarıyla).  
  *(En iyi seçenek değildir; çünkü satırları kopyalayarak veri sızıntısı yaratır.)*
- <a href="https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/">**`Synthetic Minority Oversampling Technique - SMOTE`**</a> kullanarak, mevcut gözlemleri ağırlıklandırıp yeni veri noktaları üretmek
- Buna ek olarak, çoğunluk sınıfını bir miktar küçültmek için  
  <a href="https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/">**`RandomUnderSampler`**</a> da deneyebilirsiniz

In [8]:
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)
print(f"Eğitim :{X_train}, Doğrulama: {X_val}, Test: {X_test}")

Eğitim :            Time        V1        V2        V3        V4        V5        V6  \
272615  165180.0 -3.017333  3.215950 -2.844590 -1.341856 -0.491730 -1.776197   
191231  129171.0  2.099809 -0.890100 -2.817319 -1.208673  0.845043  0.138699   
53595    46061.0  0.812352 -0.586909 -0.667514  0.962864  0.401248  0.721682   
193549  130180.0 -2.621263 -4.439432 -2.595440 -1.117193  2.489633 -2.625322   
207723  136809.0  2.227359 -1.572316 -0.371772 -1.578679 -1.593467 -0.157863   
...          ...       ...       ...       ...       ...       ...       ...   
131478   79616.0 -0.877913 -0.301831  2.735473  1.253404 -1.145942  0.068037   
239193  150000.0 -1.662279 -0.278422  2.677875  1.479724 -0.641821  0.421010   
67705    52666.0 -1.061497  0.978902  1.629268 -1.385857 -0.074805 -1.054468   
233557  147611.0  2.081836 -0.128730 -1.497688  0.049290  0.463113 -0.260199   
155997  106980.0  0.054160  1.031223  0.394136 -0.422840  0.641408 -0.982156   

              V7        V8     

In [10]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Dengeleme sonrası dolandırıcılık sayısı: {sum(y_train_res == 1)}")

Dengeleme sonrası dolandırıcılık sayısı: 181946


## 2. Sinir Ağı yinelemeleri

Sınıflarınızı yeniden dengelediğinize göre, test puanınızı optimize etmek için bir sinir ağı uyarlayın. Aşağıdaki ipuçlarını kullanmaktan çekinmeyin:

- Girişlerinizi normalleştirin!
    - Modelinizdeki ön işlemeyi “boru hattı” haline getirmek için tercihen model içinde bir [`Normalization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Normalization) katmanı kullanın. 
    - Veya modelinizin dışında sklearn'in [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) öğesini kullanın, `X_train`, `X_val` ve `X_test` öğelerini uygulayın.
- Modelinizi aşırı uyumlu hale getirin, ardından aşağıdakileri kullanarak düzenleyin:
- Erken Durdurma kriterleri
- [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) katmanları
    - veya [`düzenleyiciler`](https://www.tensorflow.org/api_docs/python/tf/keras/regularizers) katmanları
- 🚨 İzlemek istediğiniz metrikleri ve kullanmak istediğiniz kayıp fonksiyonunu dikkatlice düşünün!

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# DİKKAT: Scaler'ı sadece eğitim verisiyle 'fit' ediyoruz (Sızıntıyı önlemek için)
X_train_res = scaler.fit_transform(X_train_res)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

model = models.Sequential([
    layers.Input(shape=(X_train_res.shape[1],)),
    
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2), # Ezberlemeyi önlemek için %20'sini rastgele kapat
    
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid') # İkili sınıflandırma (0 veya 1)
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.AUC(name='prc', curve='PR') # Precision-Recall eğrisi
    ]
)

### 🧪 Kodunu Test Et

Orijinal dengesiz veri kümesinin (`X_test`, `y_test`) temsil edici bir örneğinde gerçek test performansınızın altında kalan sonuçları `precision` ve `recall` değişkenlerine kaydedin.

In [13]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

# 1. Test seti üzerinde tahminleri al (Olasılık olarak döner)
y_pred_probs = model.predict(X_test)

# 2. Olasılıkları sınıfa dönüştür (Eşik değer: 0.5)
# Not: Dolandırıcılık vakalarını kaçırmamak için bu eşik ileride ayarlanabilir.
y_pred = (y_pred_probs > 0.5).astype(int)

# 3. Precision ve Recall değerlerini hesapla ve kaydet
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"--- Gerçek Dünya (Test) Performansı ---")
print(f"Precision (Keskinlik): {precision:.4f}")
print(f"Recall (Duyarlılık): {recall:.4f}")

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
--- Gerçek Dünya (Test) Performansı ---
Precision (Keskinlik): 0.0015
Recall (Duyarlılık): 0.5408


In [14]:
from nbresult import ChallengeResult

result = ChallengeResult('solution',
    precision=precision,
    recall=recall,
    fraud_number=len(y_test[y_test == 1]),
    non_fraud_number=len(y_test[y_test == 0]),
)

result.write()
print(result.check())

## 🏁 İsteğe bağlı: Bu zorluk için Google'ın çözümünü okuyun
Bu oturumdaki tüm zorlukları tamamladığınız için tebrikler!

Son olarak, Google'ın kendi çözümünü doğrudan [Colab'da buradan](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/structured_data/imbalanced_data.ipynb) okuyun. 

İlginç teknikler ve en iyi uygulamaları keşfedeceksiniz.